In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import tqdm

from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from attention_classifier import AttentionClassifier
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo



features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3

In [3]:
import pandas as pd


df_3b = pd.read_hdf("../events/MG3/dataframes/threeTag_picoAOD.h5")
df_bg4b = pd.read_hdf("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
df_signal = pd.read_hdf("../events/MG3/dataframes/HH4b_picoAOD.h5")
df_3b["signal"] = False
df_bg4b["signal"] = False
df_signal["signal"] = True
raw_df_list = [df_3b, df_bg4b, df_signal]

In [4]:
from correct_systematic_error import constrained_linear_fit, get_histograms
from utils import get_quantiles_with_weights
from events_data import get_is_signal
from signal_region import get_SR_CR_cut
from scipy import stats

signal_filename = "HH4b_picoAOD.h5"
experiment_name = "CR_fvt_training_v2"
n_3b = 100_0000
seed = 0
signal_ratio = 0.02

hparam_filter = {
    "experiment_name": experiment_name,
    "dataset": lambda x: all([x["seed"] == seed, 
                        x["n_3b"] == n_3b, 
                        x["signal_ratio"] == signal_ratio]),
    "aux_info_step": 3, 
    "model": "FvTClassifier"
}
hashes = TrainingInfo.find(hparam_filter)
assert len(hashes) == 1

CR_fvt_tinfo = TrainingInfo.load(hashes[0])
base_encoder_hash = CR_fvt_tinfo.hparams["encoder_hash"]
base_fvt_tinfo = TrainingInfo.load(base_encoder_hash)
smeared_fvt_hash = CR_fvt_tinfo.hparams["smeared_fvt_hash"]
smeared_fvt_tinfo = TrainingInfo.load(smeared_fvt_hash)

# Use the same mother samples and exclude ones used for training base & smeared FvT model
msamples = MotherSamples.load(smeared_fvt_tinfo.ms_hash)
tst_scdinfo = msamples.scdinfo[~smeared_fvt_tinfo.ms_idx]
df_tst = tst_scdinfo.fetch_data_with_loaded_df(raw_df_list)
df_tst["signal"] = get_is_signal(tst_scdinfo, signal_filename)
events_tst = EventsData.from_dataframe(df_tst, features)

In [5]:
events_3b = EventsData.from_dataframe(df_3b, features)
events_bg4b = EventsData.from_dataframe(df_bg4b, features)
events_signal = EventsData.from_dataframe(df_signal, features)

In [25]:
events_3b.X[0]

array([108.89524078,  71.01351166,  51.83506775,  46.61869049,
         1.65891707,  -1.97747457,   1.2102381 ,   2.28751111,
         0.        ,   1.2024585 ,  -1.47929835,  -2.84688878,
         0.        ,   0.        ,   0.        ,   0.        ])

In [36]:
import cvxpy as cp

def calculate_distance(X_1, X_2):
    pT_1 = X_1[[0, 4, 8, 12]]
    pT_2 = X_2[[0, 4, 8, 12]]
    eta_1 = X_1[[1, 5, 9, 13]]
    eta_2 = X_2[[1, 5, 9, 13]]
    phi_1 = X_1[[2, 6, 10, 14]]
    phi_2 = X_2[[2, 6, 10, 14]]
    
    eta_diff = eta_1.reshape(-1, 1) - eta_2.reshape(1, -1)
    phi_diff = phi_1.reshape(-1, 1) - phi_2.reshape(1, -1)
    
    D = np.sqrt(eta_diff**2 + phi_diff**2)
    F = cp.Variable((4, 4))
    objective = cp.Minimize(cp.sum(cp.multiply(F, D)))
    constraints = [
        F >= 0,
        cp.sum(F, axis=1) <= pT_1,
        cp.sum(F, axis=0) <= pT_2,
        cp.sum(F) == np.min([pT_1.sum(), pT_2.sum()])
    ]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return prob.value * (1/np.sqrt(np.pi**2 + 25)) + np.abs(np.sum(pT_1) - np.sum(pT_2))

In [38]:
import numpy as np
import ot
from itertools import product
max_n_events = 100

for seed in range(10):
    print(f"seed = {seed}")
    rand_idx = np.random.choice(len(events_3b), max_n_events)
    events_source = events_3b.clone()[rand_idx]

    for target in ["bg4b", "signal"]:
        if target == "bg4b":
            rand_idx = np.random.choice(len(events_bg4b), max_n_events)
            events_target = events_bg4b.clone()[rand_idx]
        else:
            rand_idx = np.random.choice(len(events_signal), max_n_events)
            events_target = events_signal.clone()[rand_idx]
            
        M = np.zeros((max_n_events, max_n_events))
            
        for i, j in product(range(max_n_events), range(max_n_events)):
            M[i, j] = calculate_distance(events_source.X[i], events_target.X[j])

        source_weights = events_source.weights
        source_weights /= np.sum(source_weights)

        target_weights = events_target.weights
        target_weights /= np.sum(target_weights)

        pi_s = source_weights
        pi_t = target_weights

        M = ot.dist(events_source.X[:, [0, 4, 8, 12]], events_target.X[:, [0, 4, 8, 12]])

        out = ot.emd(pi_s, pi_t, M = M, numItermax=1e7)

        # Calculate the Wasserstein distance
        wasserstein_distance = np.sum(out * M)
        print(f"Wasserstein distance to {target}: {wasserstein_distance}")

seed = 0
Wasserstein distance to bg4b: 854.9542506879077
Wasserstein distance to signal: 813.6354028340697
seed = 1
Wasserstein distance to bg4b: 230.20898030477335
Wasserstein distance to signal: 591.1537305488388
seed = 2
Wasserstein distance to bg4b: 289.5272852329873
Wasserstein distance to signal: 576.3795353485567
seed = 3
Wasserstein distance to bg4b: 1354.007066826051
Wasserstein distance to signal: 1644.1023140349446
seed = 4
Wasserstein distance to bg4b: 1086.3026942946003
Wasserstein distance to signal: 721.3155800607223
seed = 5
Wasserstein distance to bg4b: 3079.7341608572733
Wasserstein distance to signal: 320.5149608762737
seed = 6
Wasserstein distance to bg4b: 1888.5384208762687
Wasserstein distance to signal: 196.96711985771222
seed = 7


KeyboardInterrupt: 